## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

In [1]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown

In [2]:
load_dotenv(override=True)

True

## OpenAI Hosted Tools

OpenAI Agents SDK includes the following hosted tools:

The `WebSearchTool` lets an agent search the web.  
The `FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
The `ComputerTool` allows automating computer use tasks like taking screenshots and clicking.

### Important note - API charge of WebSearchTool

This is costing me 2.5 cents per call for OpenAI WebSearchTool. That can add up to $2-$3 for the next 2 labs. We'll use free and low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern. Also student Christian W. pointed out that OpenAI can sometimes charge for multiple searches for a single call, so it could sometimes cost more than 2.5 cents per call.

Costs are here: https://platform.openai.com/docs/pricing#web-search

In [3]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [4]:
message = "Latest AI Agent frameworks in 2026"

with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

In 2026, several AI agent frameworks have emerged, each offering unique capabilities for developing autonomous systems:

- **Orchestral**: A lightweight Python framework providing a unified, type-safe interface for building LLM agents across major providers, addressing challenges like fragmented APIs and inconsistent tool-calling behaviors. ([arxiv.org](https://arxiv.org/abs/2601.02577?utm_source=openai))

- **MARS (Modular Agent with Reflective Search)**: Optimized for autonomous AI research, MARS employs budget-aware planning via cost-constrained Monte Carlo Tree Search and a modular construction pipeline to manage complex research repositories. ([arxiv.org](https://arxiv.org/abs/2602.02660?utm_source=openai))

- **Agent Lightning**: A flexible framework enabling reinforcement learning-based training of large language models for any AI agent, achieving decoupling between agent execution and training for seamless integration. ([arxiv.org](https://arxiv.org/abs/2508.03680?utm_source=openai))

- **AgentForge**: An open-source Python framework designed for constructing LLM-driven autonomous agents, featuring a composable skill abstraction and a unified LLM backend interface for seamless integration. ([arxiv.org](https://arxiv.org/abs/2601.13383?utm_source=openai))

- **OpenClaw**: An open-source autonomous AI assistant software that executes tasks via large language models, using messaging platforms as its main user interface. ([en.wikipedia.org](https://en.wikipedia.org/wiki/OpenClaw?utm_source=openai))

- **Moltbook**: An internet forum designed exclusively for AI agents, launched in January 2026, allowing AI agents to interact and collaborate. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Moltbook?utm_source=openai))

- **Agent2Agent (A2A)**: An open protocol defining how AI agents communicate across different systems, enabling interoperability between agents built by different vendors or frameworks. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Agent2Agent?utm_source=openai))

These frameworks reflect the rapid evolution of AI agent development, offering diverse tools and protocols to meet various application needs. 

### As always, take a look at the trace

https://platform.openai.com/traces

### We will now use Structured Outputs, and include a description of the fields

In [ ]:
# See note above about cost of WebSearchTool

HOW_MANY_SEARCHES = 3

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

# Use Pydantic to define the Schema of our response - this is known as "Structured Outputs"
# With massive thanks to student Wes C. for discovering and fixing a nasty bug with this!

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)

In [7]:

message = "Latest AI Agent frameworks in 2026"

with trace("Search"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)

searches=[WebSearchItem(reason='To find updated information on AI agent frameworks released or popularized in 2026.', query='latest AI agent frameworks 2026'), WebSearchItem(reason='To see analyses and comparisons of various AI frameworks that are trending in 2026.', query='AI frameworks comparison 2026'), WebSearchItem(reason='To gather expert opinions and insights on the development of AI agents as of 2026.', query='expert insights AI agents 2026')]


In [8]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("tyler.mayfield23@gmail.com") # Change this to your verified email
    to_email = To("tyler.mayfield23@gmail.com") # Change this to your email
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return "success"

In [15]:
send_email

<function __main__.send_email(report: __main__.ReportData)>

In [9]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model="gpt-4o-mini",
)



In [10]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
)

### The next 3 functions will plan and execute the search, using planner_agent and search_agent

In [18]:
plan_test = []
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    plan_test.append(result)
    print("Finished Planning")
    return result.final_output

perform_test = []
async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    perform_test.append(results)
    print("Finished searching")
    return results

search_test = []
async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    search_test.append(result)
    return result.final_output

### The next 2 functions write a report and email it

In [57]:

report_test = []
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    report_test.append(result)
    return result.final_output

email_test = []
async def send_email(report: ReportData):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    email_test.append(report)
    email_test.append(result)
    return report

### Showtime!

In [58]:
query ="Latest AI Agent frameworks in 2026"

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await send_email(report)  
    print("Hooray!")




Starting research...
Planning searches...
Will perform 3 searches
Finished Planning
Searching...
Finished searching
Thinking about report...
Finished writing report
Writing email...
Email sent
Hooray!


In [56]:
plan_test[0]

RunResult(input='Query: Latest AI Agent frameworks in 2026', new_items=[MessageOutputItem(agent=Agent(name='PlannerAgent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You are a helpful research assistant. Given a query, come up with a set of web searches to perform to best answer the query. Output 3 terms to query for.', prompt=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), input_guardrails=[], output_guardrails=[], output_type=<class '__main__.WebSearchPlan'>, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True), raw_item=ResponseOutputMessage(id='msg_0c4801035ae4f31

In [53]:
perform_test[0][0]

'As of February 2026, several AI agent frameworks have emerged, each offering unique features for developing autonomous AI systems:\n\n- **OpenClaw**: An open-source autonomous AI agent developed by Peter Steinberger, enabling users to create AI agents that interact with services like email and Spotify. ([en.wikipedia.org](https://en.wikipedia.org/wiki/OpenClaw?utm_source=openai))\n\n- **Moltbook**: Launched in January 2026 by Matt Schlicht, Moltbook is a social network exclusively for AI agents, allowing them to interact and collaborate. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Moltbook?utm_source=openai))\n\n- **AgentForge**: A lightweight, open-source Python framework designed to democratize the construction of LLM-driven autonomous agents through a modular architecture. ([arxiv.org](https://arxiv.org/abs/2601.13383?utm_source=openai))\n\n- **Orchestral**: A lightweight Python framework providing a unified, type-safe interface for building LLM agents across major providers,

In [ ]:
search_test[0]

'Search term: AI agent technology trends 2026\nReason for searching: To explore industry reports and analyses about new AI agent technologies and frameworks available in 2026.'

In [69]:
report_test[0].raw_responses[0].output[0].content[0].text

'{"short_summary":"As of February 2026, several innovative AI agent frameworks have emerged, including OpenClaw, LangChain, and MetaGPT, each designed to address unique challenges and capabilities in developing autonomous AI systems. These frameworks are enhancing functionalities across various domains, integrating multi-agent systems, and focusing on security and collaboration, thereby marking a significant evolution in AI technology.","markdown_report":"# Latest AI Agent Frameworks in 2026\\n\\nAs of February 2026, the field of artificial intelligence (AI) is witnessing rapid advancements, particularly in the development of AI agent frameworks. These frameworks are designed to manage and orchestrate autonomous AI systems, enabling them to perform complex tasks across a range of applications. Here we will explore the leading frameworks that have emerged or gained prominence this year, examining their unique capabilities, applications, and contributions to the evolution of intelligent 

In [72]:
email_test[0]

ReportData(short_summary='As of February 2026, several innovative AI agent frameworks have emerged, including OpenClaw, LangChain, and MetaGPT, each designed to address unique challenges and capabilities in developing autonomous AI systems. These frameworks are enhancing functionalities across various domains, integrating multi-agent systems, and focusing on security and collaboration, thereby marking a significant evolution in AI technology.', markdown_report="# Latest AI Agent Frameworks in 2026\n\nAs of February 2026, the field of artificial intelligence (AI) is witnessing rapid advancements, particularly in the development of AI agent frameworks. These frameworks are designed to manage and orchestrate autonomous AI systems, enabling them to perform complex tasks across a range of applications. Here we will explore the leading frameworks that have emerged or gained prominence this year, examining their unique capabilities, applications, and contributions to the evolution of intellig

### As always, take a look at the trace

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Congratulations on your progress, and a request</h2>
            <span style="color:#00cc00;">You've reached an important moment with the course; you've created a valuable Agent using one of the latest Agent frameworks. You've upskilled, and unlocked new commercial possibilities. Take a moment to celebrate your success!<br/><br/>Something I should ask you -- my editor would smack me if I didn't mention this. If you're able to rate the course on Udemy, I'd be seriously grateful: it's the most important way that Udemy decides whether to show the course to others and it makes a massive difference.<br/><br/>And another reminder to <a href="https://www.linkedin.com/in/eddonner/">connect with me on LinkedIn</a> if you wish! If you wanted to post about your progress on the course, please tag me and I'll weigh in to increase your exposure.
            </span>
        </td>
    </tr>